<a href="https://colab.research.google.com/github/urmilapol/urmilapolprojects/blob/master/vishnu2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# 1. Install prerequisites and Marathi/Devanagari fonts with libraqm support
!apt-get install -y fonts-deva libraqm-dev -qq
!pip install Pillow pandas -q

import os
import shutil
import textwrap
import pandas as pd
from PIL import Image, ImageDraw, ImageFont
from google.colab import files

# 2. Upload CSV File
print("Please upload your CSV file (Sr.No., Sanskrit Shlok, Marathi Meaning):")
uploaded = files.upload()
csv_filename = list(uploaded.keys())[0]

# Read CSV
df = pd.read_excel(csv_filename)
print(f"Loaded {len(df)} rows successfully.")

# Map column names automatically (in case column names vary slightly)
cols = df.columns
sr_col = cols[0]
shlok_col = cols[1]
meaning_col = cols[2]

# 3. Setup Directories & Devanagari Fonts
output_dir = "vishnu_shlok_images"
os.makedirs(output_dir, exist_ok=True)

# Using system installed Devanagari font (Lohit / Gargi / Samanata)
font_path = "/usr/share/fonts/truetype/fonts-deva-extra/gargi.ttf"
if not os.path.exists(font_path):
    font_path = "/usr/share/fonts/truetype/lohit-devanagari/Lohit-Devanagari.ttf"

# Load fonts for Title, Sanskrit Shloka, and Marathi Meaning
font_title = ImageFont.truetype(font_path, 34)
font_shlok = ImageFont.truetype(font_path, 36)
font_meaning = ImageFont.truetype(font_path, 26)
font_label = ImageFont.truetype(font_path, 22)

# Helper function to wrap text by pixel width
def wrap_text_by_pixels(text, font, max_width, draw):
    lines = []
    # Split manual lines or sentences first
    paragraphs = str(text).split('।')
    for p_idx, p in enumerate(paragraphs):
        p = p.strip()
        if not p:
            continue
        if p_idx < len(paragraphs) - 1:
            p += " ।"

        words = p.split()
        current_line = []
        for word in words:
            test_line = " ".join(current_line + [word])
            bbox = draw.textbbox((0, 0), test_line, font=font, language='hi')
            if bbox[2] - bbox[0] <= max_width:
                current_line.append(word)
            else:
                if current_line:
                    lines.append(" ".join(current_line))
                current_line = [word]
        if current_line:
            lines.append(" ".join(current_line))
    return lines

# 4. Generate Image for Each Shloka
IMG_WIDTH, IMG_HEIGHT = 1080, 1350  # 4:5 Portrait HD resolution

for index, row in df.iterrows():
    sr_no = str(row[sr_col]).strip()
    shlok_text = str(row[shlok_col]).strip()
    meaning_text = str(row[meaning_col]).strip()

    # Create base background canvas (Deep Divine Royal Blue & Gold theme)
    img = Image.new("RGB", (IMG_WIDTH, IMG_HEIGHT), color=(14, 23, 42))
    draw = ImageDraw.Draw(img)

    # Outer & Inner Gold Borders
    draw.rectangle([(25, 25), (IMG_WIDTH - 25, IMG_HEIGHT - 25)], outline=(212, 175, 55), width=4)
    draw.rectangle([(38, 38), (IMG_WIDTH - 38, IMG_HEIGHT - 38)], outline=(184, 134, 11), width=1)

    # Header: Title & Shloka Number
    header_text = f"॥ श्री विष्णु सहस्रनाम स्तोत्रम् ॥"
    bbox_h = draw.textbbox((0, 0), header_text, font=font_title, language='hi')
    draw.text(((IMG_WIDTH - (bbox_h[2] - bbox_h[0])) // 2, 65), header_text, font=font_title, fill=(255, 215, 0), language='hi')

    sub_header = f"श्लोक क्रमांक : {sr_no}"
    bbox_s = draw.textbbox((0, 0), sub_header, font=font_label, language='hi')
    draw.text(((IMG_WIDTH - (bbox_s[2] - bbox_s[0])) // 2, 120), sub_header, font=font_label, fill=(226, 232, 240), language='hi')

    # Decorative Divider Line
    draw.line([(100, 165), (IMG_WIDTH - 100, 165)], fill=(212, 175, 55), width=2)

    # Section 1: Sanskrit Shloka Box
    y_cursor = 210
    draw.rounded_rectangle([(70, y_cursor), (IMG_WIDTH - 70, y_cursor + 320)], radius=15, fill=(30, 41, 59), outline=(212, 175, 55), width=2)

    shlok_lines = wrap_text_by_pixels(shlok_text, font_shlok, IMG_WIDTH - 180, draw)
    shlok_start_y = y_cursor + (320 - (len(shlok_lines) * 55)) // 2

    for line in shlok_lines:
        bbox_l = draw.textbbox((0, 0), line, font=font_shlok, language='hi')
        line_w = bbox_l[2] - bbox_l[0]
        draw.text(((IMG_WIDTH - line_w) // 2, shlok_start_y), line, font=font_shlok, fill=(255, 230, 109), language='hi')
        shlok_start_y += 55

    # Section 2: Marathi Meaning Box
    y_cursor = 580
    meaning_label = "॥ मराठी भावार्थ व नामावली ॥"
    bbox_m = draw.textbbox((0, 0), meaning_label, font=font_label, language='hi')
    draw.text(((IMG_WIDTH - (bbox_m[2] - bbox_m[0])) // 2, y_cursor), meaning_label, font=font_label, fill=(212, 175, 55), language='hi')

    y_cursor += 45
    draw.rounded_rectangle([(70, y_cursor), (IMG_WIDTH - 70, IMG_HEIGHT - 90)], radius=15, fill=(24, 32, 47), outline=(71, 85, 105), width=1)

    # Wrap and draw Marathi meaning text
    meaning_lines = wrap_text_by_pixels(meaning_text, font_meaning, IMG_WIDTH - 170, draw)
    meaning_start_y = y_cursor + 35

    for line in meaning_lines:
        if meaning_start_y > IMG_HEIGHT - 130:
            break
        draw.text((100, meaning_start_y), line, font=font_meaning, fill=(248, 250, 252), language='hi')
        meaning_start_y += 42

    # Footer
    footer_text = "ॐ नमो भगवते वासुदेवाय"
    bbox_f = draw.textbbox((0, 0), footer_text, font=font_label, language='hi')
    draw.text(((IMG_WIDTH - (bbox_f[2] - bbox_f[0])) // 2, IMG_HEIGHT - 65), footer_text, font=font_label, fill=(148, 163, 184), language='hi')

    # Save Image
    safe_sr = str(sr_no).replace(" ", "_").replace(".", "")
    out_filename = os.path.join(output_dir, f"shlok_{safe_sr}.png")
    img.save(out_filename, "PNG", quality=95)

print(f"\nAll {len(df)} images generated in folder: '{output_dir}'.")

# 5. Zip and Download
zip_filename = "Vishnu_Sahasranamam_Images.zip"
shutil.make_archive("Vishnu_Sahasranamam_Images", 'zip', output_dir)
print(f"Compressed images to {zip_filename}. Initiating download...")
files.download(zip_filename)


Please upload your CSV file (Sr.No., Sanskrit Shlok, Marathi Meaning):


Saving finalVishnu_Sahasranamam.xlsx to finalVishnu_Sahasranamam (1).xlsx
Loaded 126 rows successfully.

All 126 images generated in folder: 'vishnu_shlok_images'.
Compressed images to Vishnu_Sahasranamam_Images.zip. Initiating download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>